# Ch.3 — Backpropagation, intuitively

*3Blue1Brown, Deep Learning series, Ch.3. Concise lecture notes. Ch.2 said gradient descent needs $-\nabla C$; this chapter is **what that gradient means** and **how backprop computes it** — intuition, no calculus yet (that's Ch.4).*

**Carried over:** MLP for MNIST, ~13,000 weights & biases. Learning = nudge them to minimize cost.

## Recap: what the gradient means

- Cost $C$ takes all ~13,000 weights & biases → one "badness" number, averaged over training data.
- $-\nabla C$ tells us how to nudge **every** weight & bias to reduce cost fastest.
- Each component's **magnitude = sensitivity**: how much cost changes per unit change in that weight/bias → its **relative importance**.

> Backprop = the algorithm that **computes this gradient**. Goal here: get intuition for *why* it has the form it does.

## Step 1 — Focus on a single training example

Take one image (say a **'2'**). Current output (10 activations) is junk, e.g. `[0.5, 0.8, 0.2, 1.0, ...]`.

**Desired:** push output neuron **'2' → 1.0**, all other outputs **→ 0.0**.

The *sizes* of the desired nudges are **proportional to how far off** each output currently is — the more wrong a neuron, the harder we want to push it.

## Step 2 — Three ways to nudge an output neuron

To raise one output activation $a^{(L)}_j = \sigma(\dots)$, you can change three things feeding it:

1. **Bias** $b$ — shift it directly.
2. **Weights** $w$ — increase weights on connections from the **most active** previous neurons.
   - *Biggest bang for the buck*: changes to weights from high-activation neurons matter most (weight × activation). → **"neurons that fire together, wire together"** (loose Hebbian analogy).
3. **Previous activations** $a^{(L-1)}$ — you can't set these directly, but you can *want* them changed: neurons with **positive** weights should be **more** active, negative weights **less** active.

Point 3 is the key: it expresses a **desire for changes one layer back.**

## Step 3 — Add up the demands, then propagate backward

- Every one of the 10 output neurons has its **own** wishlist for how each previous-layer activation should change.
- **Sum these wishes** (weighted by how much each output cares) → a single list of desired changes for layer $L-1$.
- That list is exactly the same kind of "desired output" we started with — so **recurse**: apply the same logic to get desired changes for layer $L-2$, and so on **back to the input**.

This backward flow of "desired nudges" through every layer = **backpropagation**. It assigns **blame**: each weight & bias is told how to change.

## Step 4 — One example isn't enough: average

- The nudges above came from **one** training image — they'd make *that* digit better but ignore all others.
- The true desired nudge for each weight/bias = **average** of the nudges demanded by **every** training example.
- That averaged set of nudges **is** (proportional to) the negative gradient $-\nabla C$.

$$-\nabla C \;\propto\; \frac{1}{N}\sum_{k=1}^{N}(\text{nudges from example }k)$$

## Step 5 — Stochastic Gradient Descent (SGD)

Averaging over **all** N examples for **every** step is far too slow.

**Fix:** shuffle the data, split into **mini-batches** (e.g. 100 examples). Each step:
- compute the gradient on **one mini-batch** → an **approximation** of the true gradient,
- take a step, move to the next batch.

- Each step is **cheap & fast**; the path **zig-zags** downhill instead of taking the exact straightest route.
- Analogy: a **drunk man stumbling quickly downhill** beats a careful man calculating each exact step.
- This is **SGD** — the standard way real networks are trained.

In [ ]:
import numpy as np

# Illustrate the backprop intuition on a single neuron + the SGD averaging idea.
# Desired-nudge sizes are proportional to how wrong each output is.
output  = np.array([0.5, 0.8, 0.2, 1.0, 0.3, 0.6, 0.1, 0.0, 0.4, 0.7])
target  = np.zeros(10); target[2] = 1.0          # this image is a '2'

error          = output - target                 # how wrong each output neuron is
desired_nudge  = -error                           # push opposite the error
print("nudge on neuron '2':", round(desired_nudge[2], 2), "(big positive -> raise it)")
print("nudge on neuron '3':", round(desired_nudge[3], 2), "(negative -> suppress it)")

# Weight update favors connections from the most active previous neurons (w x a):
prev_act       = np.array([0.9, 0.1, 0.7])        # previous-layer activations
weight_nudge   = desired_nudge[2] * prev_act      # 'fire together, wire together'
print("weight nudges into '2':", np.round(weight_nudge, 2), "-> active neuron #0 gets biggest")

# SGD: gradient of a mini-batch approximates the full-data gradient.
rng        = np.random.default_rng(0)
full_grad  = rng.normal(size=1000)                # pretend: true gradient over all data
batch_grad = full_grad[:100]                      # one mini-batch estimate
print("full-data mean:", round(full_grad.mean(), 3),
      "| mini-batch mean:", round(batch_grad.mean(), 3), "(noisy but cheap)")

## Why this matters

- Backprop **distributes blame**: it efficiently turns "the output was wrong" into a precise "**this** weight should go up, **that** bias down" for all ~13,000 parameters.
- It's just the **chain rule applied recursively**, organized so each layer reuses the next layer's result — the explicit calculus is **Ch.4**.
- Combined with SGD, it's what makes training large networks **computationally feasible**.

**References**
- Michael Nielsen, *Neural Networks and Deep Learning* — Ch. on backprop.
- 3B1B **Ch.4** — backpropagation calculus (the chain-rule details).